# WixQA — Production RAG Pipeline

## Overview
End-to-end Retrieval-Augmented Generation (RAG) pipeline built on the
[WixQA](https://huggingface.co/datasets/Wix/WixQA) benchmark.

| Phase | Purpose |
|-------|---------|
| **Setup & Auth** | Clone repo, install deps, set API keys |
| **Baseline Pipeline** | Chunk → Embed → Index → Retrieve → Generate |
| **Development (Synthetic)** | Tune retrieval/chunk config on WixQA-Synthetic |
| **Evaluation (Expert / Simulated)** | Final metrics on held-out splits |
| **Advanced RAG** | Reranking, query rewriting, hybrid retrieval |
| **Results Table** | Comparison across configurations |

---
## 0. Environment Setup

In [ ]:
repo = "wixqa-rag"
import os
import sys

from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

!git clone https://{GITHUB_TOKEN}@github.com/bencejdanko/{repo}.git
os.chdir(f'/content/{repo}')
!git pull
sys.path.append(f"/content/{repo}")

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
# ── API Keys ──────────────────────────────────────────────────────────────
# Store OPENROUTER_API_KEY in Colab Secrets (key icon on the left sidebar)
from google.colab import userdata
import openai

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

client = openai.OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

# Quick sanity check
test = client.chat.completions.create(
    model="google/gemini-2.0-flash-001",
    messages=[{"role": "user", "content": "Say 'RAG pipeline ready' and nothing else."}],
    max_tokens=10,
)
print(test.choices[0].message.content)

---
## 1. Load WixQA Dataset (HuggingFace)

In [ ]:
from src.data_loader import load_kb_corpus, load_qa_split, build_kb_lookup, qa_to_dataframe

# Knowledge base — 6,221 articles
articles = load_kb_corpus()

# QA splits
qa_synthetic   = load_qa_split("synthetic")   # Development / tuning
qa_expert      = load_qa_split("expert")       # Evaluation ONLY
qa_simulated   = load_qa_split("simulated")    # Evaluation ONLY

print(f"\nKB articles   : {len(articles):,}")
print(f"QA synthetic  : {len(qa_synthetic):,} (dev/tuning only)")
print(f"QA expert     : {len(qa_expert):,} (evaluation)")
print(f"QA simulated  : {len(qa_simulated):,} (evaluation)")

In [ ]:
# Quick peek at the data
import pandas as pd
df_synth = qa_to_dataframe(qa_synthetic)
df_synth.head(3)

In [ ]:
# Sample article
print("Sample article fields:", list(articles[0].keys()))
print("\nArticle ID  :", articles[0]["id"])
print("Type        :", articles[0]["article_type"])
print("URL         :", articles[0]["url"])
print("Content len :", len(articles[0]["contents"]))

---
## 2. Baseline RAG Pipeline

### Baseline Configuration

| Component | Configuration Used |
|-----------|--------------------|
| Chunk size / overlap | 512 / 64 chars |
| Chunk strategy | Recursive character splitting |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector database | FAISS — `IndexFlatIP` (exact cosine) |
| Retriever type | Dense (bi-encoder) |
| Reranker | None (baseline) |
| Query rewriting | None (baseline) |
| Generator model | `google/gemini-2.0-flash-001` |
| Prompting strategy | Basic (retrieval → answer) |

In [ ]:
from src.pipeline import RAGPipeline

baseline_pipeline = RAGPipeline(
    articles=articles,
    client=client,
    # Chunking
    chunk_strategy="recursive",
    chunk_size=512,
    chunk_overlap=64,
    # Embedding
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    # Retrieval
    retriever_type="dense",
    top_k_retrieval=10,
    # No reranking / query rewriting at baseline
    use_reranker=False,
    query_strategy="none",
    # Generation
    generator_model="google/gemini-2.0-flash-001",
    gen_strategy="basic",
    max_context_chars=6000,
)

baseline_pipeline.print_config()

In [ ]:
# ── Quick sanity inference ─────────────────────────────────────────────────
sample = qa_synthetic[0]
result = baseline_pipeline.run(sample["question"])

print("Question  :", sample["question"])
print("-" * 60)
print("Reference :", sample["answer"])
print("-" * 60)
print("Generated :", result["answer"])
print("-" * 60)
print(f"Retrieved : {len(result['retrieved'])} chunks from articles: "
      f"{list({c.article_id for c,_ in result['retrieved']})}")

---
## 3. Development Evaluation on WixQA-Synthetic

Use this split **only** for tuning hyperparameters (chunk size, top-k, etc.).

In [ ]:
from src.evaluator import evaluate_dataset
import pandas as pd

# ── Baseline on Synthetic (dev set, 100 samples for speed) ────────────────
DEV_SAMPLES = 100  # increase to 6221 for full evaluation

dev_results, dev_agg = evaluate_dataset(
    qa_rows=qa_synthetic,
    retriever=baseline_pipeline.retriever,
    generator=baseline_pipeline.generator,
    client=client,
    use_llm_judge=False,           # flip to True to enable LLM judge
    top_k=10,
    k_values=[1, 3, 5, 10],
    max_samples=DEV_SAMPLES,
)

print("\n── Baseline Development Results (WixQA-Synthetic) ──")
for metric, value in sorted(dev_agg.items()):
    print(f"  {metric:<25} {value:.4f}")

In [ ]:
# Preview sample predictions
pd.DataFrame(dev_results)[["question", "rouge1_f", "rouge2_f", "rougeL_f",
                            "recall@5", "recall@10", "mrr"]].head(10)

---
## 4. Chunking Strategy Ablation (Dev Set)

Compare fixed, recursive, and sentence-boundary chunking.

In [ ]:
from src.chunker import chunk_articles, describe_chunks
from src.embedder import Embedder
from src.vector_store import VectorStore
from src.retriever import DenseRetriever
from src.generator import Generator
from src.evaluator import evaluate_dataset
import pandas as pd

ABLATION_SAMPLES = 50
ablation_results = []

configs = [
    dict(strategy="fixed",     chunk_size=256, overlap=32),
    dict(strategy="fixed",     chunk_size=512, overlap=64),
    dict(strategy="fixed",     chunk_size=1024, overlap=128),
    dict(strategy="recursive", chunk_size=256, overlap=32),
    dict(strategy="recursive", chunk_size=512, overlap=64),
    dict(strategy="recursive", chunk_size=1024, overlap=128),
    dict(strategy="sentence",  chunk_size=512, overlap=64),
]

# Shared embedder for efficiency
shared_embedder = Embedder(model_name="sentence-transformers/all-MiniLM-L6-v2")
shared_generator = Generator(client=client, model="google/gemini-2.0-flash-001",
                              strategy="basic")

for cfg in configs:
    label = f"{cfg['strategy']}-{cfg['chunk_size']}/{cfg['overlap']}"
    print(f"\n{'='*60}")
    print(f"Testing: {label}")

    chunks = chunk_articles(
        articles, strategy=cfg["strategy"],
        chunk_size=cfg["chunk_size"], overlap=cfg["overlap"]
    )
    stats = describe_chunks(chunks)
    embeddings = shared_embedder.encode([c.text for c in chunks])

    vs = VectorStore(dim=shared_embedder.dim, index_type="flat")
    vs.add(chunks, embeddings)

    retriever = DenseRetriever(shared_embedder, vs, top_k=10)

    _, agg = evaluate_dataset(
        qa_rows=qa_synthetic,
        retriever=retriever,
        generator=shared_generator,
        use_llm_judge=False,
        top_k=10,
        k_values=[1, 5, 10],
        max_samples=ABLATION_SAMPLES,
        verbose=False,
    )
    ablation_results.append({
        "config": label,
        "n_chunks": stats["total_chunks"],
        "avg_chars": stats["avg_chars"],
        **{k: round(v, 4) for k, v in agg.items()},
    })
    print(f"  recall@5={agg.get('recall@5',0):.4f}  "
          f"recall@10={agg.get('recall@10',0):.4f}  "
          f"rouge1={agg.get('rouge1_f',0):.4f}")

abl_df = pd.DataFrame(ablation_results)
print("\n── Chunking Ablation Summary ──")
abl_df[["config", "n_chunks", "avg_chars",
        "recall@1", "recall@5", "recall@10",
        "mrr", "rouge1_f", "rouge2_f", "rougeL_f"]]

---
## 5. Advanced Retrieval: Reranking & Hybrid Search (Dev Set)

In [ ]:
# ── 5a: Cross-encoder reranking ───────────────────────────────────────────
from src.reranker import CrossEncoderReranker
from src.retriever import DenseRetriever

# Use best chunking config from ablation (default: recursive-512/64)
reranker = CrossEncoderReranker(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5,
)

# Wrap baseline retriever: retrieve 20 candidates, rerank to top 5
class RerankedRetriever:
    def __init__(self, base_retriever, reranker, top_k=5):
        self.base = base_retriever
        self.reranker = reranker
        self.top_k = top_k
    def retrieve(self, query, top_k=None):
        k = top_k or self.top_k
        candidates = self.base.retrieve(query, top_k=k * 4)
        return self.reranker.rerank(query, candidates, top_n=k)

reranked_retriever = RerankedRetriever(
    base_retriever=baseline_pipeline.retriever,
    reranker=reranker,
    top_k=5
)

_, rerank_agg = evaluate_dataset(
    qa_rows=qa_synthetic,
    retriever=reranked_retriever,
    generator=baseline_pipeline.generator,
    use_llm_judge=False,
    top_k=5,
    k_values=[1, 3, 5],
    max_samples=50,
    verbose=True,
)
print("\n── With Cross-Encoder Reranking (dev) ──")
for k, v in sorted(rerank_agg.items()):
    print(f"  {k:<25} {v:.4f}")

In [ ]:
# ── 5b: Hybrid retrieval (Dense + BM25 RRF) ───────────────────────────────
from src.retriever import BM25Retriever, HybridRetriever

sparse_retriever = BM25Retriever(baseline_pipeline.chunks)
hybrid_retriever = HybridRetriever(
    dense=baseline_pipeline.retriever,
    sparse=sparse_retriever,
    alpha=0.7,
    top_k=10,
)

_, hybrid_agg = evaluate_dataset(
    qa_rows=qa_synthetic,
    retriever=hybrid_retriever,
    generator=baseline_pipeline.generator,
    use_llm_judge=False,
    top_k=10,
    k_values=[1, 3, 5, 10],
    max_samples=50,
    verbose=True,
)
print("\n── Hybrid Retrieval (RRF, alpha=0.7, dev) ──")
for k, v in sorted(hybrid_agg.items()):
    print(f"  {k:<25} {v:.4f}")

---
## 6. Query Rewriting Strategies (Dev Set)

In [ ]:
from src.query_rewriter import QueryRewriter
from src.evaluator import evaluate_dataset

DEV_QR_SAMPLES = 30
qr_results = []

for qr_strategy in ["none", "step_back", "hyd", "multi_query"]:
    print(f"\nTesting query rewriting: {qr_strategy}")

    qr = QueryRewriter(strategy=qr_strategy, client=client,
                       model="google/gemini-2.0-flash-001")

    class QRRetriever:
        """Wraps rewriter + base retriever."""
        def __init__(self, base, rewriter, top_k=10):
            self.base = base
            self.rewriter = rewriter
            self.top_k = top_k

        def retrieve(self, query, top_k=None):
            k = top_k or self.top_k
            queries = self.rewriter.rewrite(query)
            seen = {}
            for q in queries:
                for chunk, score in self.base.retrieve(q, top_k=k):
                    if chunk.chunk_id not in seen or score > seen[chunk.chunk_id][1]:
                        seen[chunk.chunk_id] = (chunk, score)
            return sorted(seen.values(), key=lambda x: x[1], reverse=True)[:k]

    qr_retriever = QRRetriever(baseline_pipeline.retriever, qr)

    _, agg = evaluate_dataset(
        qa_rows=qa_synthetic,
        retriever=qr_retriever,
        generator=baseline_pipeline.generator,
        use_llm_judge=False,
        top_k=10,
        k_values=[1, 5, 10],
        max_samples=DEV_QR_SAMPLES,
        verbose=False,
    )
    qr_results.append({"strategy": qr_strategy, **{k: round(v, 4) for k, v in agg.items()}})
    print(f"  recall@5={agg.get('recall@5',0):.4f}  rouge1={agg.get('rouge1_f',0):.4f}")

import pandas as pd
qr_df = pd.DataFrame(qr_results)
print("\n── Query Rewriting Ablation (dev) ──")
qr_df[["strategy", "recall@1", "recall@5", "recall@10", "mrr", "rouge1_f", "rougeL_f"]]

---
## 7. Prompting Strategy Ablation (Dev Set)

In [ ]:
from src.generator import Generator
from src.evaluator import evaluate_dataset
import pandas as pd

prompt_results = []
DEV_PROMPT_SAMPLES = 30

for ps in ["basic", "cot", "cite", "compression"]:
    print(f"\nPrompting strategy: {ps}")
    gen = Generator(client=client, model="google/gemini-2.0-flash-001",
                    strategy=ps)
    _, agg = evaluate_dataset(
        qa_rows=qa_synthetic,
        retriever=baseline_pipeline.retriever,
        generator=gen,
        use_llm_judge=False,
        top_k=10,
        k_values=[5],
        max_samples=DEV_PROMPT_SAMPLES,
        verbose=False,
    )
    prompt_results.append({"prompting_strategy": ps,
                            **{k: round(v, 4) for k, v in agg.items()}})
    print(f"  rouge1={agg.get('rouge1_f',0):.4f}  "
          f"rouge2={agg.get('rouge2_f',0):.4f}  "
          f"rougeL={agg.get('rougeL_f',0):.4f}")

prompt_df = pd.DataFrame(prompt_results)
print("\n── Prompting Strategy Ablation (dev) ──")
prompt_df[["prompting_strategy", "rouge1_f", "rouge2_f", "rougeL_f", "exact_match"]]

---
## 8. Best Pipeline Configuration

Assemble the best configuration identified on the dev set.

In [ ]:
# ── Re-build the best pipeline based on dev ablation ─────────────────────
# Adjust the parameters below based on your ablation results.

best_pipeline = RAGPipeline(
    articles=articles,
    client=client,
    # Chunking — best from ablation
    chunk_strategy="recursive",
    chunk_size=512,
    chunk_overlap=64,
    # Embedding
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    # Retrieval
    retriever_type="hybrid",
    top_k_retrieval=20,
    # Reranking
    use_reranker=True,
    reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_k_rerank=5,
    # Query rewriting — best from QR ablation
    query_strategy="step_back",
    # Generation
    generator_model="google/gemini-2.0-flash-001",
    gen_strategy="cot",
    max_context_chars=6000,
)

best_pipeline.print_config()

---
## 9. Final Evaluation — WixQA-ExpertWritten & WixQA-Simulated

> ⚠️ **Run these cells only once** — final held-out evaluation.  
> Do NOT use these splits for further tuning.

In [ ]:
# ── 9a: WixQA-ExpertWritten ───────────────────────────────────────────────
print("=" * 60)
print("FINAL EVALUATION — WixQA-ExpertWritten (N=200)")
print("=" * 60)

expert_results, expert_agg = evaluate_dataset(
    qa_rows=qa_expert,
    retriever=best_pipeline.retriever,
    generator=best_pipeline.generator,
    client=client,
    model="google/gemini-2.0-flash-001",
    use_llm_judge=True,      # Enable LLM judge for final evaluation
    top_k=5,
    k_values=[1, 3, 5, 10],
    verbose=True,
)

print("\n── ExpertWritten Results ──")
for metric, value in sorted(expert_agg.items()):
    print(f"  {metric:<25} {value:.4f}")

In [ ]:
# ── 9b: WixQA-Simulated ───────────────────────────────────────────────────
print("=" * 60)
print("FINAL EVALUATION — WixQA-Simulated (N=200)")
print("=" * 60)

simulated_results, simulated_agg = evaluate_dataset(
    qa_rows=qa_simulated,
    retriever=best_pipeline.retriever,
    generator=best_pipeline.generator,
    client=client,
    model="google/gemini-2.0-flash-001",
    use_llm_judge=True,
    top_k=5,
    k_values=[1, 3, 5, 10],
    verbose=True,
)

print("\n── Simulated Results ──")
for metric, value in sorted(simulated_agg.items()):
    print(f"  {metric:<25} {value:.4f}")

---
## 10. Comparative Results Table

In [ ]:
import pandas as pd

# ── Baseline dev results ──────────────────────────────────────────────────
# (populated from cells above)
results_table = pd.DataFrame([
    {
        "System": "Baseline (dense, basic prompt)",
        "Split": "Synthetic (dev)",
        "Recall@5":   dev_agg.get("recall@5", float("nan")),
        "Recall@10":  dev_agg.get("recall@10", float("nan")),
        "MRR":        dev_agg.get("mrr", float("nan")),
        "ROUGE-1":    dev_agg.get("rouge1_f", float("nan")),
        "ROUGE-L":    dev_agg.get("rougeL_f", float("nan")),
        "EM":         dev_agg.get("exact_match", float("nan")),
        "LLM-Factuality":    float("nan"),
        "LLM-Completeness":  float("nan"),
    },
    {
        "System": "Best Pipeline (hybrid+rerank+CoT)",
        "Split": "ExpertWritten (test)",
        "Recall@5":   expert_agg.get("recall@5", float("nan")),
        "Recall@10":  expert_agg.get("recall@10", float("nan")),
        "MRR":        expert_agg.get("mrr", float("nan")),
        "ROUGE-1":    expert_agg.get("rouge1_f", float("nan")),
        "ROUGE-L":    expert_agg.get("rougeL_f", float("nan")),
        "EM":         expert_agg.get("exact_match", float("nan")),
        "LLM-Factuality":   expert_agg.get("llm_factuality", float("nan")),
        "LLM-Completeness": expert_agg.get("llm_completeness", float("nan")),
    },
    {
        "System": "Best Pipeline (hybrid+rerank+CoT)",
        "Split": "Simulated (test)",
        "Recall@5":   simulated_agg.get("recall@5", float("nan")),
        "Recall@10":  simulated_agg.get("recall@10", float("nan")),
        "MRR":        simulated_agg.get("mrr", float("nan")),
        "ROUGE-1":    simulated_agg.get("rouge1_f", float("nan")),
        "ROUGE-L":    simulated_agg.get("rougeL_f", float("nan")),
        "EM":         simulated_agg.get("exact_match", float("nan")),
        "LLM-Factuality":   simulated_agg.get("llm_factuality", float("nan")),
        "LLM-Completeness": simulated_agg.get("llm_completeness", float("nan")),
    },
])

float_cols = [c for c in results_table.columns if c not in ["System", "Split"]]
results_table[float_cols] = results_table[float_cols].applymap(lambda x: round(x, 4))

print("\n" + "=" * 100)
print("FINAL COMPARATIVE RESULTS")
print("=" * 100)
print(results_table.to_string(index=False))

---
## 11. Baseline Configuration Summary Table

| Component | Configuration Used |
|-----------|--------------------|
| **Chunk size / overlap** | 512 chars / 64 chars |
| **Chunk strategy** | Recursive character splitting |
| **Embedding model** | `sentence-transformers/all-MiniLM-L6-v2` (384-dim) |
| **Vector database** | FAISS `IndexFlatIP` (exact inner product / cosine) |
| **Retriever type** | Dense bi-encoder retrieval |
| **Reranker** | None (cross-encoder added in advanced pipeline) |
| **Query rewriting** | None (step-back / HyDE tested in advanced pipeline) |
| **Generator model** | `google/gemini-2.0-flash-001` via OpenRouter |
| **Prompting strategy** | Basic (retrieval → answer, system-restricted to context) |

In [ ]:
# Print live config from the pipeline object
baseline_pipeline.print_config()